# 08. Hyperparameter tuning con Optuna

**Fases del guía metodológica cubiertas: 14 (Hyperparameter tuning y experimentación)**

> Regla central: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.



## 14.1 Qué ajustamos

Sobre los dos modelos más fuertes de la fase 06 tuneamos:

- **LightGBM**: `n_estimators, max_depth, learning_rate, subsample, colsample_bytree,
  min_child_samples, reg_alpha, reg_lambda`.
- **RandomForest** (runner-up): `n_estimators, max_depth, min_samples_leaf, max_features,
  min_samples_split, class_weight`.

## 14.2 Método

**Optuna** (TPE sampler) con **StratifiedKFold(5)** sobre **train**.
El test permanece bloqueado. La métrica de optimización es ROC-AUC (primaria).

### 14.2.1 Preparación

Cargamos train con feature engineering. Optuna probará combinaciones de hiperparámetros;
cada trial ejecuta una CV de 5 folds (rápida en este dataset de 396 filas de train). Fijamos la
semilla del sampler para que la búsqueda sea reproducible.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
d = load_processed()
Xtr = add_domain_features(d["X_train"]); ytr = d["y_train"]
print("Train:", Xtr.shape, "| prevalencia:", round(ytr.mean(), 3))


C:\Users\sgml1\Desktop\student-alcohol-consumption\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (396, 37) | prevalencia: 0.394



### 14.2.2 Optimización de LightGBM

Definimos la función objetivo: dado un trial, se construye un `LGBMClassifier` con los
hiperparámetros sugeridos (en escalas adecuadas: logarítmica para learning rate y
regularización), se envuelve en el pipeline seguro y se devuelve la media de ROC-AUC de
la CV. Optuna (TPE) explora el espacio durante 40 trials, guardando el mejor valor y los
mejores parámetros. Al terminar, guardamos esos parámetros en
`configs/best_params_lgbm.json` (se congelarán en la fase 16).


In [2]:

from src.models.train_model import make_pipeline, get_preprocessor
from sklearn.model_selection import cross_val_score, StratifiedKFold
import lightgbm as lgb
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        "class_weight": "balanced",
        "random_state": 42, "n_jobs": -1, "verbose": -1,
    }
    model = lgb.LGBMClassifier(**params)
    pipe = make_pipeline(model)
    return cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=1).mean()

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_lgbm, n_trials=40, show_progress_bar=False)
print("LightGBM -> Mejor ROC-AUC CV:", round(study.best_value, 4))
print("LightGBM -> Mejores hiperparámetros:", study.best_params)

# Guardar mejores parámetros de LightGBM (se congelan en fase 16)
best_lgbm = study.best_params
best_lgbm["random_state"] = 42; best_lgbm["class_weight"] = "balanced"
import json
(ROOT / "configs" / "best_params_lgbm.json").write_text(json.dumps(best_lgbm, indent=2), encoding="utf-8")
print("Parámetros LightGBM guardados en configs/best_params_lgbm.json")


LightGBM -> Mejor ROC-AUC CV: 0.8498
LightGBM -> Mejores hiperparámetros: {'n_estimators': 450, 'max_depth': 3, 'learning_rate': 0.04982463018185643, 'subsample': 0.6995779135100955, 'colsample_bytree': 0.7412725184966888, 'min_child_samples': 33, 'reg_alpha': 0.8502878122682971, 'reg_lambda': 0.0019393188935196738}
Parámetros LightGBM guardados en configs/best_params_lgbm.json



### 14.2.3 Optimización de RandomForest

Repetimos el proceso con **RandomForest**, incluyendo `class_weight` como hiperparámetro
candidato (los árboles pueden beneficiarse de balancear las clases). Guardamos los mejores
parámetros en `configs/best_params_rf.json`. En la fase 15 se compararán los modelos
tuneados entre sí; el ganador será el modelo final.

> **Nota de decisión del proyecto**: tras la auditoría se fijó `class_weight="balanced"`
> como configuración final del modelo (comprobado: +0.011 ROC-AUC CV frente a `None`,
> y coherente con la documentación). Si Optuna propone `None`, se sobreescribe con
> `"balanced"` al congelar los parámetros.


In [3]:

# Tuning de RandomForest (runner-up) con el mismo protocolo
from sklearn.ensemble import RandomForestClassifier

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", ["balanced", None]),
        "random_state": 42, "n_jobs": -1,
    }
    model = RandomForestClassifier(**params)
    pipe = make_pipeline(model)
    return cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=1).mean()

study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=25, show_progress_bar=False)
print("RandomForest -> Mejor ROC-AUC CV:", round(study_rf.best_value, 4))
print("RandomForest -> Mejores hiperparámetros:", study_rf.best_params)

best_rf = study_rf.best_params
best_rf["random_state"] = 42
# Decisión del proyecto: balancear clases (verificado: +0.011 ROC-AUC CV)
best_rf["class_weight"] = "balanced"
(ROOT / "configs" / "best_params_rf.json").write_text(json.dumps(best_rf, indent=2), encoding="utf-8")
print("Parámetros RandomForest guardados en configs/best_params_rf.json")
print("class_weight final fijado a 'balanced' (decisión de coherencia documentada)")


RandomForest -> Mejor ROC-AUC CV: 0.847
RandomForest -> Mejores hiperparámetros: {'n_estimators': 550, 'max_depth': 9, 'min_samples_leaf': 3, 'min_samples_split': 2, 'max_features': 0.5284109559755278, 'class_weight': None}
Parámetros RandomForest guardados en configs/best_params_rf.json
class_weight final fijado a 'balanced' (decisión de coherencia documentada)



### 14.2.4 Historial de optimización

Dibujamos la evolución del mejor valor de ROC-AUC a lo largo de los trials de ambos
estudios. Una curva que mejora rápido y luego se aplana indica que la búsqueda ha
convergido; si aún subiera al final, podríamos ampliar el presupuesto. La figura se
guarda en `reports/figures/08_optuna_history.png`.


In [4]:

# Historial de optimización
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([t.value for t in study.trials], marker="o", ms=3, label="LightGBM")
ax.plot([t.value for t in study_rf.trials], marker="s", ms=3, label="RandomForest")
ax.set_xlabel("Trial"); ax.set_ylabel("ROC-AUC CV"); ax.set_title("Optimización Optuna")
ax.legend()
plt.tight_layout(); plt.savefig(ROOT / "reports" / "figures" / "08_optuna_history.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_19288\790871532.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
